In [48]:
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [49]:
from openai import OpenAI
import requests
from minsearch import Index
import os

In [50]:
api_key=os.environ["PERPLEXITY_API_KEY"]
base_url="https://api.perplexity.ai"

In [51]:
openai_client = OpenAI(api_key=api_key, base_url=base_url)

In [43]:
docs_url = "https://datatalks.club/faq/json/courses.json"

In [7]:
response = requests.get(docs_url)

In [10]:
courses_raw = response.json()

In [13]:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}"""

    course_response = requests.get(course_url)
    course_response.raise_for_status()                  #if sth is broken, create an error,
                                                        #don't continue
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1350

In [12]:
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [17]:
index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

In [18]:
index.fit(documents)

In [ ]:
question = 'I just discovered the course. Can I join now?'

In [26]:
search_results = index.search(
    question,
    boost_dict={'question': 2.0, 'section': 0.5},
    filter_dict={'course': 'llm-zoomcamp'},
    num_results=5
)

search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [22]:
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'How should I start the course and follow the weekly workflow?',
 'When will the course be offered next?']

In [28]:
def search(question, course="llm-zoompcamp"):
    boost_dict = {"question": 2.0, "section": 0.5}
    filter_dict = {"course": course}

    return index.search(
        question,
        boost_dict={'question': 2.0, 'section': 0.5},
        filter_dict={'course': 'llm-zoomcamp'},
        num_results=5
    )


In [29]:
search_results = search(question)

In [30]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [31]:
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
"""

In [32]:
USER_PROMPT_TEMPLATE = """
Question:
{question}

Context:
{context}
"""

In [33]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("")

    return "\n".join(lines).strip()

In [34]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [35]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [54]:
response =  openai_client.chat.completions.create(
    model="sonar",
        messages=[
        {"role": "user", "content": prompt}
    ]
)

In [ ]:
response.choices[0].message.content

'Yes, you can join the course now[1]. However, if you want to receive a **certificate**, you must submit your project while the course is still accepting submissions[1].\n\nYou do not need to wait for a confirmation email or registration approval to start; you are already accepted and can begin learning and submitting homework immediately as long as the form for participation is open[1]. Registration is primarily used to gauge interest before the course start date, not to verify your eligibility[1].\n\nTo get started:\n- Follow the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp-2026/)[1].\n- Watch lesson videos, work through notebooks/code, review homework instructions on GitHub, and submit answers via the course platform before deadlines[1].\n\nNote: Certificates are only awarded for 

In [72]:
response.usage.cost

{'input_tokens_cost': 0.00048,
 'output_tokens_cost': 0.0003,
 'request_cost': 0.005,
 'total_cost': 0.00577}

In [60]:
response.model_dump_json(indent = 2)

'{\n  "id": "4f72266c-1f35-4565-a7c6-f9bcaa7de589",\n  "choices": [\n    {\n      "finish_reason": "stop",\n      "index": 0,\n      "logprobs": null,\n      "message": {\n        "content": "Yes, you can join the course now[1]. However, if you want to receive a **certificate**, you must submit your project while the course is still accepting submissions[1].\\n\\nYou do not need to wait for a confirmation email or registration approval to start; you are already accepted and can begin learning and submitting homework immediately as long as the form for participation is open[1]. Registration is primarily used to gauge interest before the course start date, not to verify your eligibility[1].\\n\\nTo get started:\\n- Follow the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp-2026/)[1].\\n- 

In [ ]:
def llm(instructions, user_prompt, model="gpt-5.4-mini"):
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = openai_client.chat.completions.create(
        model=model,
        input=message_history
    )

    return response.choices[0].message.content

In [75]:
def rag(query, model="gpt-5.4-mini"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [76]:
answer = rag("I just discovered the course. Can I join now?")
print(answer)

NotFoundError: Error code: 404